# Setup Validation

Run this notebook after installation to verify everything works:
1. tribev2 imports correctly
2. Model weights download and load
3. A basic prediction runs end-to-end
4. Brain surface visualization works

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

## 1. Check imports

In [ ]:
import numpy as np
import pandas as pd
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import nilearn
print(f"nilearn: {nilearn.__version__}")

from tribev2 import TribeModel
print("\ntribev2 imported successfully!")

## 2. Load model

In [ ]:
from core.model import load_model

model = load_model()
print("Model loaded successfully!")

## 3. Test prediction with a sample stimulus

Replace the path below with any short video/audio/text file you have.

In [ ]:
# Option A: Test with a video file
# video_path = str(PROJECT_ROOT / "stimuli/videos/test_clip.mp4")
# df = model.get_events_dataframe(video_path=video_path)

# Option B: Test with a text file
# text_path = str(PROJECT_ROOT / "stimuli/text/test_sentence.txt")
# df = model.get_events_dataframe(text_path=text_path)

# Uncomment one of the above and run:
# preds, segments = model.predict(events=df)
# print(f"Predictions shape: {preds.shape}")
# print(f"  {preds.shape[0]} timepoints x {preds.shape[1]} vertices")
# print(f"  Value range: [{preds.min():.4f}, {preds.max():.4f}]")

print("Uncomment a stimulus path above and re-run this cell.")

## 4. Test visualization

In [ ]:
from nilearn import datasets

fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")
print(f"fsaverage5 surface loaded")
print(f"  Left pial: {fsaverage['pial_left']}")
print(f"  Right pial: {fsaverage['pial_right']}")

# Quick test: plot random data on the brain
from core.viz import plot_brain_surface

random_data = np.random.randn(20_484) * 0.5
fig = plot_brain_surface(
    random_data,
    title="Test: random noise on fsaverage5",
    cmap="cold_hot",
    threshold=0.3,
)
print("\nVisualization works!")

## 5. Test core modules

In [ ]:
from core.stimuli import build_event_related_design, build_block_design
from core.glm import fit_first_level_glm
from core.contrasts import compute_contrast

# Synthetic test: fake 2-condition experiment
n_timepoints, n_vertices = 300, 100
fake_preds = np.random.randn(n_timepoints, n_vertices)

fake_events = pd.DataFrame([
    {"onset": t, "duration": 1.0, "condition": "A" if i % 2 == 0 else "B"}
    for i, t in enumerate(range(10, 290, 14))
])

glm_result = fit_first_level_glm(fake_preds, fake_events)
print(f"GLM labels: {glm_result['labels']}")
print(f"Betas shape: {glm_result['betas'].shape}")

contrast = compute_contrast(glm_result, "A - B")
print(f"Contrast shape: {contrast.shape}")
print(f"Contrast range: [{contrast.min():.4f}, {contrast.max():.4f}]")

print("\nAll core modules work!")

## Summary

If all cells above ran without errors, your setup is complete. Next steps:

1. Place stimulus files in `stimuli/` or `experiments/humor/stimuli/`
2. Run your first experiment: `python experiments/humor/run.py`
3. Explore results in `experiments/humor/notebooks/explore.ipynb`